# Session 4 — Agentic RAG: the LLM decides when to retrieve

**Goal**: convert your RAG chain into a **RAG agent**. Retrieval becomes a *tool* the LLM can call — zero, one, or multiple times — instead of a fixed pipeline step.

**Mental model recap**:
- **Chain** (session 2–3): question → *always* retrieve → prompt → answer. One LLM call. Deterministic.
- **Agent** (today): question → LLM *decides* → maybe retrieve (maybe several times, maybe reformulating the query) → answer. Multiple LLM calls. Flexible.

**Prerequisites**:
- `uv add langgraph`
- Ollama running (app open or `ollama serve`)
- Session 2: `document.pdf`.

> **Note on the model**: `llama3.2` supports tool calling, which is what makes this possible with a local model. Small models are less reliable at tool use than big ones — observing *where it fails* is part of today's learning.

## 1. Load the vector store (same as always)

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings,
    collection_name="pdf_docs",   # the AWS whitepaper collection from session 3
)
print(f"Vectors: {db._collection.count()}")

retriever = db.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectors: 715


## 2. Wrap the retriever as a *tool*

This is the key conceptual step. A **tool** is just a function with a name and a description — the LLM reads the description to decide whether calling it would help answer the user.

The docstring is not decoration: **it's the prompt that teaches the model when to use the tool.**

In [3]:
from langchain_core.tools import tool

@tool
def search_aws_docs(query: str) -> str:
    """Search the AWS Well-Architected Data Analytics Lens whitepaper.
    Use this tool for any question about AWS analytics services, data architecture
    best practices, security, Glue, Redshift, EMR, Athena, or Lake Formation.
    Input should be a focused search query."""
    docs = retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)

# Tools are inspectable — this is what the LLM "sees":
print(search_aws_docs.name)
print(search_aws_docs.description)

search_aws_docs
Search the AWS Well-Architected Data Analytics Lens whitepaper.
    Use this tool for any question about AWS analytics services, data architecture
    best practices, security, Glue, Redshift, EMR, Athena, or Lake Formation.
    Input should be a focused search query.


## 3. Build the agent

We use LangGraph's prebuilt ReAct agent — the standard reasoning loop: **Reason → Act (call tool) → Observe (read result) → repeat until ready to answer**.

In [6]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

llm = ChatOllama(model="llama3.2", temperature=0)

agent = create_react_agent(
    model=llm,
    tools=[search_aws_docs],
    prompt=(
        "You are a helpful assistant answering questions about AWS data analytics. "
        "Use the search tool when the question requires information from the whitepaper. "
        "If the question is general chit-chat, answer directly without searching."
    ),
)

/var/folders/9g/m2fp6bjs1gs7cx_byglbc3p80000gn/T/ipykernel_76330/320937394.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 4. Watch it decide

Run two very different inputs and inspect the message trace. The interesting part is not the answer — it's **whether and how the agent chose to retrieve**.

In [7]:
# Case A: chit-chat — the agent should NOT search
result = agent.invoke({"messages": [("user", "Hi! How are you today?")]})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hi! How are you today?
================================== Ai Message ==================================
Tool Calls:
  search_aws_docs (0000d078-2825-4248-93ac-9e377727f905)
 Call ID: 0000d078-2825-4248-93ac-9e377727f905
  Args:
    query: AWS data analytics services
================================= Tool Message =================================
Name: search_aws_docs

AWS Well-Architected Framework
Data Analytics Lens
Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.

Data Analytics Lens AWS Well-Architected Framework
You can view and optimize your costs through the AWS Cost and Usage Report and the Cost and 
Usage Dashboards Operations Solution (CUDOS) reports.
Best practice 12.2 – Build local or build centralized data analytics platforms
Teams can establish their own data analytics resources that support their analytical needs locally, 
rather than extracting informati

In [8]:
# Case B: domain question — the agent SHOULD search
result = agent.invoke({"messages": [("user", "What are the best practices for securing data in Redshift?")]})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What are the best practices for securing data in Redshift?
================================== Ai Message ==================================
Tool Calls:
  search_aws_docs (81e9a0cc-d1ec-4491-96c6-1f31bf35ac2b)
 Call ID: 81e9a0cc-d1ec-4491-96c6-1f31bf35ac2b
  Args:
    query: Redshift security best practices
================================= Tool Message =================================
Name: search_aws_docs

• WS Big Data Blog: Federating single sign-on access to your Amazon Redshift cluster with 
PingIdentity
• Amazon EMR Management Guide: Allow AWS IAM Identity Center for Amazon EMR Studio
Best practice 4.3 – Implement the required data access authorization models
User authorization determines what actions that a user is permitted to take on the data or 
resource. The data owners should be able to use the authorization methods to protect their data

• Amazon Redshift Database Developer Guide: Managing d

### What to look for in the trace

- **AIMessage with `tool_calls`** — the model decided to search, and *chose the query itself* (compare it with your original question — did it reformulate?)
- **ToolMessage** — the chunks that came back from Chroma
- **Final AIMessage** — the grounded answer

That decision step is exactly what a chain doesn't have.

## 5. Multi-step retrieval

Ask something that needs *two different searches*. A chain retrieves once with the raw question; an agent can decompose.

In [9]:
result = agent.invoke({"messages": [(
    "user",
    "Compare the security best practices for Redshift with those for EMR."
)]})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Compare the security best practices for Redshift with those for EMR.
================================== Ai Message ==================================
Tool Calls:
  search_aws_docs (21441fdf-2220-4abb-b860-f4bbe412c275)
 Call ID: 21441fdf-2220-4abb-b860-f4bbe412c275
  Args:
    query: Redshift vs EMR security best practices
================================= Tool Message =================================
Name: search_aws_docs

• WS Big Data Blog: Federating single sign-on access to your Amazon Redshift cluster with 
PingIdentity
• Amazon EMR Management Guide: Allow AWS IAM Identity Center for Amazon EMR Studio
Best practice 4.3 – Implement the required data access authorization models
User authorization determines what actions that a user is permitted to take on the data or 
resource. The data owners should be able to use the authorization methods to protect their data

• Amazon Redshift Database Developer 

### Observation exercise

Did the agent make **one search or two**? Small models often try to answer from a single retrieval. If it only searched once, try nudging via the system prompt: *"For comparison questions, search for each topic separately."* — then re-run. That's agent prompt engineering.

## 6. Stress test: when agents fail

Try these and note the failure modes (this is senior-level understanding — knowing *limits*, not just happy paths):

In [10]:
# A question that looks domain-related but is NOT in the whitepaper
result = agent.invoke({"messages": [(
    "user",
    "What is the pricing of Redshift Serverless per RPU-hour?"
)]})
print(result["messages"][-1].content)

I apologize for the detour earlier. To answer your original question directly:

The pricing of Amazon Redshift Serverless is based on RPU (Resource Utilization Percentage) hours. According to the AWS documentation, the pricing for Redshift Serverless is as follows:

* $0.025 per hour for 1-10 RPU
* $0.020 per hour for 11-50 RPU
* $0.015 per hour for 51-100 RPU
* $0.010 per hour for 101+ RPU

Please note that these prices are subject to change, and you should check the AWS website for the most up-to-date pricing information.

Also, keep in mind that Redshift Serverless is a pay-as-you-go model, so you only pay for the actual time your cluster is running. This can help optimize costs for infrequent or unpredictable workloads.


In [11]:
# Ambiguous — will it search or hallucinate?
result = agent.invoke({"messages": [("user", "Is EMR better than Glue?")]})
print(result["messages"][-1].content)

Based on the search results, it appears that both EMR and Glue have their own strengths and weaknesses.

EMR (Elastic MapReduce) is a fully managed service that allows you to run Hadoop, Spark, and other big data processing workloads on Amazon Web Services (AWS). It's suitable for frequently running jobs that require semipersistent data storage. However, it can be more expensive than Glue, especially for infrequent or intermittent use cases.

Glue, on the other hand, is a fully managed extract, transform, and load (ETL) service that makes it easy to prepare and load data for analysis. It's suitable for less complex ETL workloads and provides a cost-effective option compared to EMR. Glue also offers serverless and server-based architectures, making it more flexible than EMR.

In summary, if you need to run frequently running jobs with semipersistent data storage, EMR might be a better choice. However, if you have less complex ETL workloads or want a cost-effective option with flexibilit